# Bloc 01 - Plan your trip with Kayak 

## 01 - Collecte de données

Import de librairies

In [ ]:
import json
import os
from time import sleep

import boto3
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display

load_dotenv()

True

In [19]:
# Variables d'environnement
OPENWEATHERMAP_KEY = os.environ["OPENWEATHERMAP_KEY"]
AWS_ACCESS_KEY_ID = os.environ["AWS_ACCESS_KEY_ID"]
AWS_SECRET_ACCESS_KEY = os.environ["AWS_SECRET_ACCESS_KEY"]

bucket_name = os.environ["BUCKET_NAME"]

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name="eu-west-3"
)

In [20]:
cities = [
    "Mont Saint Michel",
    "St Malo",
    "Bayeux",
    "Le Havre",
    "Rouen",
    "Paris",
    "Amiens",
    "Lille",
    "Strasbourg",
    "Chateau du Haut Koenigsbourg",
    "Colmar",
    "Eguisheim",
    "Besancon",
    "Dijon",
    "Annecy",
    "Grenoble",
    "Lyon",
    "Gorges du Verdon",
    "Bormes les Mimosas",
    "Cassis",
    "Marseille",
    "Aix en Provence",
    "Avignon",
    "Uzes",
    "Nimes",
    "Aigues Mortes",
    "Saintes Maries de la mer",
    "Collioure",
    "Carcassonne",
    "Ariege",
    "Toulouse",
    "Montauban",
    "Biarritz",
    "Bayonne",
    "La Rochelle"
]

### Récupération des données des villes

In [21]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:147.0) Gecko/20100101 Firefox/147.0',
    "Accept-Language": "fr,fr-FR;q=0.9,en-US;q=0.8,en;q=0.7"
}

nominatim_cities_filename = "../data/cities.json"
nominatim_base_url = "https://nominatim.openstreetmap.org/"
search_city_url = "search?format=json&limit=1"
nominatim_cities_data = []

if os.path.exists(nominatim_cities_filename):
    with open(nominatim_cities_filename, "r", encoding="utf-8") as f:
        nominatim_cities_data = json.load(f)
        
else:
    for city in cities:
        response_city = requests.get(f"{nominatim_base_url}{search_city_url}&q={city},france", headers=headers).json()
        response_city[0]["city"] = city
        nominatim_cities_data.append(response_city[0])
        sleep(1)

    with open(nominatim_cities_filename, "w", encoding="utf-8") as f:
        json.dump(nominatim_cities_data, f, ensure_ascii=False, indent=4)

print(nominatim_cities_data)

[{'place_id': 262508722, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'way', 'osm_id': 211285890, 'lat': '48.6359541', 'lon': '-1.5114600', 'class': 'place', 'type': 'islet', 'place_rank': 20, 'importance': 0.4723710249003365, 'addresstype': 'islet', 'name': 'Mont Saint-Michel', 'display_name': 'Mont Saint-Michel, Le Mont-Saint-Michel, Avranches, Manche, Normandie, France métropolitaine, 50170, France', 'boundingbox': ['48.6349172', '48.6370310', '-1.5133292', '-1.5094796'], 'city': 'Mont Saint Michel'}, {'place_id': 261963228, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'relation', 'osm_id': 905534, 'lat': '48.6495180', 'lon': '-2.0260409', 'class': 'boundary', 'type': 'administrative', 'place_rank': 16, 'importance': 0.6250994319975981, 'addresstype': 'town', 'name': 'Saint-Malo', 'display_name': 'Saint-Malo, Ille-et-Vilaine, Bretagne, France métropolitaine, 35400, France', 'boundin

In [22]:
cities_coord = { city["city"]: {"lat": city["lat"], "lon": city["lon"]} for city in nominatim_cities_data }
print(cities_coord)

{'Mont Saint Michel': {'lat': '48.6359541', 'lon': '-1.5114600'}, 'St Malo': {'lat': '48.6495180', 'lon': '-2.0260409'}, 'Bayeux': {'lat': '49.2764624', 'lon': '-0.7024738'}, 'Le Havre': {'lat': '49.4938975', 'lon': '0.1079732'}, 'Rouen': {'lat': '49.4404591', 'lon': '1.0939658'}, 'Paris': {'lat': '48.8534951', 'lon': '2.3483915'}, 'Amiens': {'lat': '49.8941708', 'lon': '2.2956951'}, 'Lille': {'lat': '50.6365654', 'lon': '3.0635282'}, 'Strasbourg': {'lat': '48.5846140', 'lon': '7.7507127'}, 'Chateau du Haut Koenigsbourg': {'lat': '48.2493820', 'lon': '7.3439412'}, 'Colmar': {'lat': '48.0777517', 'lon': '7.3579641'}, 'Eguisheim': {'lat': '48.0447968', 'lon': '7.3079618'}, 'Besancon': {'lat': '47.2380222', 'lon': '6.0243622'}, 'Dijon': {'lat': '47.3215806', 'lon': '5.0414701'}, 'Annecy': {'lat': '45.8992348', 'lon': '6.1288847'}, 'Grenoble': {'lat': '45.1875602', 'lon': '5.7357819'}, 'Lyon': {'lat': '45.7578137', 'lon': '4.8320114'}, 'Gorges du Verdon': {'lat': '43.7496562', 'lon': '6.32

### Récupération des données de météo

In [23]:
openweathermap_base_url = "https://api.openweathermap.org/data/3.0/onecall"
openweathermap_filename = "../data/openweathermap_data.json"

if os.path.exists(openweathermap_filename):
    with open(openweathermap_filename, "r", encoding="utf-8") as f:
        cities_coord = json.load(f)

else:
    for k, v in cities_coord.items():
        lat = v["lat"]
        lon = v["lon"]
        
        openweathermap_coord_url = f"?lat={lat}&lon={lon}&units=metric&exclude=current,minutely,hourly,alerts&APPID={OPENWEATHERMAP_KEY}"
        cities_coord[k].update(requests.get(f"{openweathermap_base_url}{openweathermap_coord_url}").json())

    with open(openweathermap_filename, "w", encoding="utf-8") as f:
        json.dump(cities_coord, f, ensure_ascii=False, indent=4)

In [24]:
print(f"nb villes : {len(cities_coord)}")

display(cities_coord)


nb villes : 35


{'Mont Saint Michel': {'lat': 48.636,
  'lon': -1.5115,
  'timezone': 'Europe/Paris',
  'timezone_offset': 3600,
  'daily': [{'dt': 1772625600,
    'sunrise': 1772606547,
    'sunset': 1772646809,
    'moonrise': 1772652180,
    'moonset': 1772607420,
    'moon_phase': 0.53,
    'summary': 'You can expect partly cloudy in the morning, with clearing in the afternoon',
    'temp': {'day': 15.49,
     'min': 7.53,
     'max': 16.11,
     'night': 9.38,
     'eve': 11.38,
     'morn': 7.57},
    'feels_like': {'day': 14.87, 'night': 7.6, 'eve': 10.79, 'morn': 5.86},
    'pressure': 1022,
    'humidity': 68,
    'dew_point': 9.38,
    'wind_speed': 4.74,
    'wind_deg': 140,
    'wind_gust': 7.17,
    'weather': [{'id': 800,
      'main': 'Clear',
      'description': 'clear sky',
      'icon': '01d'}],
    'clouds': 10,
    'pop': 0,
    'uvi': 2.06},
   {'dt': 1772712000,
    'sunrise': 1772692827,
    'sunset': 1772733302,
    'moonrise': 1772742960,
    'moonset': 1772694660,
    'moon_

### Traitement du dataframe

In [25]:
df = pd.DataFrame(cities_coord)

df.head()


,Mont Saint Michel,St Malo,Bayeux,Le Havre,Rouen,Paris,Amiens,Lille,Strasbourg,Chateau du Haut Koenigsbourg,...,Aigues Mortes,Saintes Maries de la mer,Collioure,Carcassonne,Ariege,Toulouse,Montauban,Biarritz,Bayonne,La Rochelle
lat,48.636,48.6495,49.2765,49.4939,49.4405,48.8535,49.8942,50.6366,48.5846,48.2494,...,43.5662,43.4516,42.5251,43.213,42.9455,43.6045,44.0176,43.4833,43.4945,46.1597
lon,-1.5115,-2.026,-0.7025,0.108,1.094,2.3484,2.2957,3.0635,7.7507,7.3439,...,4.1915,4.4277,3.0832,2.3491,1.4066,1.4442,1.355,-1.5593,-1.4737,-1.1516
timezone,Europe/Paris,Europe/Paris,Europe/Paris,Europe/Paris,Europe/Paris,Europe/Paris,Europe/Paris,Europe/Paris,Europe/Paris,Europe/Paris,...,Europe/Paris,Europe/Paris,Europe/Paris,Europe/Paris,Europe/Paris,Europe/Paris,Europe/Paris,Europe/Paris,Europe/Paris,Europe/Paris
timezone_offset,3600,3600,3600,3600,3600,3600,3600,3600,3600,3600,...,3600,3600,3600,3600,3600,3600,3600,3600,3600,3600
daily,"[{'dt': 1772625600, 'sunrise': 1772606547, 'su...","[{'dt': 1772625600, 'sunrise': 1772606671, 'su...","[{'dt': 1772625600, 'sunrise': 1772606390, 'su...","[{'dt': 1772625600, 'sunrise': 1772606208, 'su...","[{'dt': 1772625600, 'sunrise': 1772605969, 'su...","[{'dt': 1772625600, 'sunrise': 1772605634, 'su...","[{'dt': 1772625600, 'sunrise': 1772605708, 'su...","[{'dt': 1772622000, 'sunrise': 1772605569, 'su...","[{'dt': 1772622000, 'sunrise': 1772604324, 'su...","[{'dt': 1772622000, 'sunrise': 1772604403, 'su...",...,"[{'dt': 1772622000, 'sunrise': 1772604918, 'su...","[{'dt': 1772622000, 'sunrise': 1772604856, 'su...","[{'dt': 1772622000, 'sunrise': 1772605135, 'su...","[{'dt': 1772625600, 'sunrise': 1772605343, 'su...","[{'dt': 1772625600, 'sunrise': 1772605556, 'su...","[{'dt': 1772625600, 'sunrise': 1772605578, 'su...","[{'dt': 1772625600, 'sunrise': 1772605619, 'su...","[{'dt': 1772625600, 'sunrise': 1772606293, 'su...","[{'dt': 1772625600, 'sunrise': 1772606273, 'su...","[{'dt': 1772625600, 'sunrise': 1772606327, 'su..."


In [26]:
data = df.copy()

In [29]:
icon_score = {
    "01": 5,  # clair
    "02": 4,  # peu nuageux
    "03": 3,  # nuages
    "04": 2,  # couvert
    "09": 1,  # pluie
    "10": 1,  # pluie
    "11": 0,  # orage
    "13": 1,  # neige
    "50": 0   # brouillard
}

cities_data = []

# supression de la structure imbriquee daily
for city in data.columns:
    df_city = pd.json_normalize(df[city]["daily"])
    df_city["lat"] = df[city]["lat"]
    df_city["lon"] = df[city]["lon"]
    df_city["city"] = city
    # df_city["city_id"] = uuid.uuid4()

    cities_data.append(df_city[1:])

df_daily = pd.concat(cities_data, ignore_index=True)

# supression de la structure imbriquee weather
df_weather = pd.json_normalize(df_daily["weather"].explode())
df_weather.index = df_daily.index

df_daily = df_daily.drop(columns=["weather"]).join(df_weather.add_prefix("weather."))

df_daily["date"] = pd.to_datetime(df_daily["dt"], unit="s")
df_daily["icon_score"] = df_daily["weather.icon"].str[:2].map(icon_score)

# Calcul des scores journaliers
df_daily["daily_score"] = 0

temp = df_daily["temp.day"]
rain = df_daily.get("rain", 0)
snow = df_daily.get("snow", 0)

score_choice = [
    2,
    -3,
    -2,
    -1
]

temp_cond = [
    (18 < temp) & (temp < 26),
    (0 < temp) | (temp > 48),
    (8 < temp) | (temp > 32),
    (18 <= temp) | (temp >= 26)
]

rain_cond = [
    rain == 0,
    rain > 10,
    rain > 5,
    rain > 0
]

snow_cond = [
    snow == 0,
    snow > 10,
    snow > 5,
    snow > 0
]

df_daily["daily_score"] = df_daily.get("daily_score", 0) + np.select(temp_cond, score_choice, default=0)
df_daily["daily_score"] = df_daily.get("daily_score", 0) + np.select(rain_cond, score_choice, default=0)
df_daily["daily_score"] = df_daily.get("daily_score", 0) + np.select(snow_cond, score_choice, default=0)

df_daily.loc[df_daily["clouds"] < 30, "daily_score"] += 1
df_daily.loc[df_daily["wind_speed"] < 7, "daily_score"] += 1
df_daily["daily_score"] += df_daily["icon_score"]

# Calcul du score par ville, on prend la moyenne de daily_score
score_par_ville = df_daily.groupby("city")["daily_score"].mean().sort_values(ascending=False)

df_daily["city_score"] = df_daily["city"].map(score_par_ville)

# Normalisation du score entre 0 et 1
all_villes_score = df_daily[["city", "lat", "lon", "city_score"]].drop_duplicates()
all_villes_score["score_normalized"] = (all_villes_score["city_score"] - all_villes_score["city_score"].min()) / (all_villes_score["city_score"].max() - all_villes_score["city_score"].min())
all_villes_score.sort_values(by=["city_score", "city",], ascending=[False, True], inplace=True)

score_normalized_par_ville = all_villes_score.set_index("city")["score_normalized"]

df_daily["city_score_normalized"] = df_daily["city"].map(score_normalized_par_ville)

df_daily.sort_values(by=["city_score", "city", "dt"], ascending=[False, True, True], inplace=True)

df_daily.to_csv("../data/openweathermap_data.csv", index=False)
all_villes_score.to_csv("../data/cities.csv", index=False)

with pd.option_context(
    "display.max_columns", None, 
    "display.max_rows", None,):
    print(df_daily)


             dt     sunrise      sunset    moonrise     moonset  moon_phase  \
56   1772708400  1772690604  1772731079  1772740620  1772692440        0.57   
57   1772794800  1772776883  1772817572  1772831280  1772779680        0.60   
58   1772881200  1772863162  1772904064  1772922060  1772867040        0.63   
59   1772967600  1772949440  1772990556           0  1772954580        0.66   
60   1773054000  1773035717  1773077047  1773012720  1773042420        0.69   
61   1773140400  1773121994  1773163539  1773103260  1773130680        0.72   
62   1773226800  1773208271  1773250030  1773193440  1773219540        0.75   
21   1772712000  1772692485  1772732867  1772742600  1772694240        0.57   
22   1772798400  1772778760  1772819363  1772833380  1772781420        0.60   
23   1772884800  1772865035  1772905858  1772924160  1772868720        0.63   
24   1772971200  1772951310  1772992354           0  1772956200        0.66   
25   1773057600  1773037584  1773078849  1773014880 

In [30]:
print(all_villes_score)

                             city      lat     lon  city_score  \
56                     Strasbourg  48.5846  7.7507    2.428571   
21                       Le Havre  49.4939  0.1080    1.857143   
49                          Lille  50.6366  3.0635    1.857143   
35                          Paris  48.8535  2.3484    1.857143   
42                         Amiens  49.8942  2.2957    1.714286   
70                         Colmar  48.0778  7.3580    1.714286   
28                          Rouen  49.4405  1.0940    1.714286   
84                       Besancon  47.2380  6.0244    1.571429   
91                          Dijon  47.3216  5.0415    1.571429   
98                         Annecy  45.8992  6.1289    1.428571   
63   Chateau du Haut Koenigsbourg  48.2494  7.3439    1.428571   
77                      Eguisheim  48.0448  7.3080    1.428571   
14                         Bayeux  49.2765 -0.7025    1.285714   
119              Gorges du Verdon  43.7497  6.3286    1.142857   
203       

## Scrape Booking.com

Installer les navigateurs nécessaires au fonctionnement de playwright 
```sh
playwright install
```

Exécuter la commande suivante dans un terminal à la racine du dossier kayak/scrap_booking
```sh
scrapy crawl scrap_booking -O ../data/scrap_booking.csv
```

Le fichier csv généré se trouve dans /data/scrap_booking.csv  
Les logs se trouvent dans le fichier <dossier_kayak_scrap_booking>/output/booking.log